# Generador de clientes sintéticos pensionarios de México

Crea una base de personas ficticias (IMSS Ley 73 / Ley 97, ISSSTE Cuentas Individuales e independientes sin afiliación) a partir de los CSV del directorio de entrada. No utiliza ni produce datos personales reales.

- **Parámetros:** todo lo ajustable (rutas, semilla, `N_CLIN`, fecha de corte, proporción por perfil, distribuciones de edad, salario, saldos, probabilidades y montos legales de referencia) vive en `config/parametros.py`.
- **Lógica:** paquete `generador/`. Cada cliente se genera por perfil, se clasifica con reglas pensionarias y se valida al 100 %.
- `N_CLIN` es exactamente el número de clientes a generar.
- Los importes están en pesos constantes de la fecha de corte. Todas las resoluciones son simuladas.

In [22]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Funciona si Jupyter se abre desde la raíz del proyecto o desde esta carpeta.
RUTA_NOTEBOOK = next(
    (ruta for ruta in (Path.cwd(), Path.cwd() / 'SyntethicDataGenerator') if (ruta / 'config').is_dir()),
    None,
)
if RUTA_NOTEBOOK is None:
    raise FileNotFoundError('No se encontró el directorio config junto al notebook.')
sys.path.insert(0, str(RUTA_NOTEBOOK))

# Descarta módulos en caché para que los cambios en config/ se apliquen al volver a ejecutar.
for modulo in [m for m in sys.modules if m.split('.')[0] in ('config', 'generador')]:
    del sys.modules[modulo]

import config as cfg
from generador import generar_clientes

cfg.DIR_OUTPUT.mkdir(exist_ok=True)
rng = np.random.default_rng(cfg.SEMILLA)

In [23]:
nombres = pd.read_csv(cfg.ARCHIVO_NOMBRES, encoding=cfg.CODIFICACION_ENTRADA)
apellidos = pd.read_csv(cfg.ARCHIVO_APELLIDOS, encoding=cfg.CODIFICACION_ENTRADA)

# Controles de calidad de los insumos.
assert len(nombres) == cfg.NOMBRES_ESPERADOS, f'Se esperan {cfg.NOMBRES_ESPERADOS} nombres.'
assert set(nombres['genero']) == set(cfg.GENEROS_ARCHIVO), 'Géneros inesperados en el archivo de nombres.'
assert len(apellidos) == cfg.APELLIDOS_ESPERADOS, f'Se esperan {cfg.APELLIDOS_ESPERADOS} apellidos.'

display(nombres.groupby('genero').size().rename('cantidad'))
display(apellidos.head())

genero
Hombre    250
Mujer     250
Name: cantidad, dtype: int64

,id,apellido
0,1,Hernández
1,2,García
2,3,Martínez
3,4,López
4,5,González


In [24]:
# Genera historial laboral, saldos (salario, RCV, AV, vivienda), pensión estimada e identificadores.
clientes = generar_clientes(rng, nombres, apellidos)
clientes.head(10)

,id_cliente,nombre,ap_paterno,ap_materno,age,f_nacim,genero,entidad,nss,num_issste,...,saldo_rcv,saldo_ahorro_solidario,porcentaje_ahorro_solidario,saldo_bono_pension,saldo_av,saldo_vivienda,av_mensual,regimen_estimado,ley_bandera,escenario_estimado
0,1,Fernanda,Nava,Arce,52,1974-03-04,M,JC,None,55987481656,...,657195.19,11880.15,1.807706,350000.0,0.00,393972.28,0.0,ISSSTE,LEY_ISSSTE,ISSSTE_PROYECCION
1,2,María Julia,Esquivel,Barragán,48,1978-08-26,M,CM,70237822013,None,...,112976.32,0.00,NaN,0.0,145093.81,74195.00,0.0,IMSS97,LEY_97,IMSS97_PROYECCION
2,3,Alison,Cisneros,Ceja,54,1972-10-10,M,AS,09917211113,None,...,712999.47,0.00,NaN,0.0,135679.08,0.00,0.0,IMSS73,LEY_73,IMSS73_PROYECCION
3,4,Emely,Castillo,Maldonado,39,1987-09-22,M,GR,66188732235,None,...,263863.68,0.00,NaN,0.0,0.00,180295.76,0.0,IMSS97,LEY_97,IMSS97_PROYECCION
4,5,Lucía,Alvarado,Toledo,31,1995-09-14,M,CL,None,None,...,0.00,0.00,NaN,0.0,0.00,0.00,0.0,NO_APLICA,NO_APLICA,INDEPENDIENTES
5,6,Melanie,Cabrera,Téllez,45,1980-12-03,M,QT,None,05008043639,...,574316.89,10898.83,1.897703,350000.0,0.00,266562.90,0.0,ISSSTE,LEY_ISSSTE,ISSSTE_PROYECCION
6,7,Esther,Barrientos,Espinoza,56,1970-05-14,M,BC,87897049703,None,...,1038890.77,0.00,NaN,0.0,0.00,0.00,0.0,IMSS73,LEY_73,IMSS73_PROYECCION
7,8,Isabella,Hinojosa,Rivera,54,1972-10-14,M,MC,64927273306,None,...,743540.46,0.00,NaN,0.0,0.00,332870.23,0.0,IMSS73,LEY_73,IMSS73_PROYECCION
8,9,Ramón,Quintero,Enríquez,50,1975-12-14,H,CH,39167513223,None,...,122668.76,0.00,NaN,0.0,0.00,78603.91,0.0,IMSS97,LEY_97,IMSS97_PROYECCION
9,10,Perla,Blanco,Grijalva,57,1968-12-29,M,SP,84006862167,None,...,613971.25,0.00,NaN,0.0,0.00,673963.91,0.0,IMSS97,LEY_97,IMSS97_PROYECCION


In [25]:
# Control de calidad del layout final y de los identificadores sintéticos.
assert list(clientes.columns) == cfg.COLUMNAS_SALIDA
assert clientes['id_cliente'].is_unique
assert clientes['curp'].is_unique
assert clientes['correo'].is_unique
assert len(clientes) == cfg.N_CLIN
assert clientes['correo'].str.endswith(('@yahoo.com', '@gmail.com', '@hotmail.com', '@outlook.com')).all()
assert clientes.loc[clientes['nss'].notna(), 'nss'].str.fullmatch(r'\d{11}').all()
assert clientes['anios_cotizados'].ge(0).all()

print('Régimen y escenario estimado')
display(clientes.groupby(['regimen_estimado', 'escenario_estimado']).size().rename('clientes'))
display(pd.crosstab(pd.cut(clientes['age'], [0, 49, 59, 64, 69, 120], labels=['<50', '50-59', '60-64', '65-69', '70+']),
                    clientes['regimen_estimado']))
display(clientes[['age', 'anios_cotizados', 'sem_cot', 'salario_mensual', 'saldo_rcv', 'saldo_av', 'saldo_vivienda', 'av_mensual']].describe(percentiles=[.5, .9, .99]).round(1))
print(f"Clientes con AV = 0: {(clientes['saldo_av'] == 0).mean():.1%} | con vivienda = 0: {(clientes['saldo_vivienda'] == 0).mean():.1%}")

Régimen y escenario estimado


regimen_estimado  escenario_estimado        
IMSS73            IMSS73_DERECHO_PENSION          80
                  IMSS73_GARANTIZADA              55
                  IMSS73_NEGATIVA_PENSION         92
                  IMSS73_PROYECCION             2873
IMSS97            IMSS97_NEGATIVA_PENSION         29
                  IMSS97_PENSION_GARANTIZADA      26
                  IMSS97_PROYECCION             3240
                  IMSS97_RECURSOS_PROPIOS          5
ISSSTE            ISSSTE_NEGATIVA_PENSION         15
                  ISSSTE_PENSION_GARANTIZADA     169
                  ISSSTE_PROYECCION             2384
                  ISSSTE_RECURSOS_PROPIOS         32
NO_APLICA         INDEPENDIENTES                1000
Name: clientes, dtype: int64

regimen_estimado,IMSS73,IMSS97,ISSSTE,NO_APLICA
age,,,,
<50,212,1723,938,741
50-59,2661,1517,1446,219
60-64,227,60,216,40


,age,anios_cotizados,sem_cot,salario_mensual,saldo_rcv,saldo_av,saldo_vivienda,av_mensual
count,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0
mean,50.3,15.2,813.9,20690.7,331165.6,10083.0,91538.0,26.2
std,7.9,10.2,535.8,11539.1,305773.6,62480.6,123198.6,138.1
min,18.0,0.0,0.0,9583.8,0.0,0.0,0.0,0.0
50%,52.0,16.0,876.0,17518.4,267929.0,0.0,39192.2,0.0
90%,58.1,28.0,1475.0,34782.7,755136.5,8313.3,263357.6,0.0
99%,63.0,32.0,1715.0,62426.1,1337121.4,223115.1,516758.4,789.1
max,64.0,38.0,2003.0,80861.3,2719421.2,1790630.8,803546.8,1806.4


Clientes con AV = 0: 87.8% | con vivienda = 0: 38.2%


In [26]:
marca_tiempo = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S_%f')
carpeta_salida = cfg.DIR_OUTPUT / marca_tiempo
carpeta_salida.mkdir(parents=True, exist_ok=False)

archivos_por_regimen = {
    'IMSS97': 'clientes_IMSS97.csv',
    'IMSS73': 'clientes_IMSS73.csv',
    'ISSSTE': 'clientes_ISSSTE_Cuentas_Individuales.csv',
}
for regimen, nombre_archivo in archivos_por_regimen.items():
    grupo = clientes.loc[clientes['regimen_estimado'].eq(regimen)]
    ruta_archivo = carpeta_salida / nombre_archivo
    grupo.to_csv(ruta_archivo, index=False, encoding=cfg.CODIFICACION_SALIDA, date_format=cfg.FORMATO_FECHA)
    print(f'{len(grupo):,} registros: {ruta_archivo.resolve()}')

independientes = clientes['regimen_estimado'].eq(cfg.NO_APLICA).sum()
print(f'{independientes:,} independientes sin régimen contributivo no incluidos en los tres archivos.')

3,300 registros: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/20261022_140742_789740/clientes_IMSS97.csv
3,100 registros: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/20261022_140742_789740/clientes_IMSS73.csv
2,600 registros: /Users/alejandro_aac/Documents/WebScrapping/SyntethicDataGenerator/DataOutput/20261022_140742_789740/clientes_ISSSTE_Cuentas_Individuales.csv
1,000 independientes sin régimen contributivo no incluidos en los tres archivos.
